In [ ]:
import pandas as pd
from io import StringIO

BATCH_CSV = """feature_a,feature_b,feature_c,target
5.1,3.5,1.0,0
4.9,3.0,1.0,0
4.7,3.2,1.0,0
6.4,3.2,1.0,1
6.9,3.1,1.0,1
5.5,2.3,1.0,1
5.0,,1.0,0
6.0,3.0,1.0,1
"""

def report_data_quality(df, expected_features=4, min_class_share=0.05):
    report = {}

    # TODO: check 1 - no missing values anywhere in df; record passed + detail
    missing_count =   df.isna().sum()
    total_missing  =  int(missing_count .sum())
    if total_missing == '0':
        report["no_missing_value"] = {
            "passed":True,
            "detail":"No missing values found "
        }
    else:
        cols_with_missing = missing_count[missing_count > 0].index.tolist()
        report["no_missing_values"] = {
            "passed": False,
            "detail": f"{total_missing} missing value(s) found in column(s): {cols_with_missing}"
        }



    # 2) Exact expected feature count
    actual_features = df.shape[1]
    if actual_features == expected_features:
        report["feature_count"] = {
            "passed": True,
            "detail": f"{actual_features} features found, {expected_features} expected"
        }
    else:
        report["feature_count"] = {
            "passed": False,
            "detail": f"{actual_features} features found, {expected_features} expected"
        }


    # TODO: check 3 - target column has exactly 2 unique classes; record passed + detail
    target_unique = df["target"].dropna().nunique()
    if target_unique == 2:
        report["binary_target"] = {
            "passed": True,
            "detail": f"target has {target_unique} unique classes"
        }
    else:
        report["binary_target"] = {
            "passed": False,
            "detail": f"target has {target_unique} unique class(es), expected 2"
        }


    # TODO: check 4 - no class proportion falls below min_class_share; record passed + detail
    class_proportions = df['target'].value_counts(normalize=True,dropna=True)
    min_share =  float (class_proportions.min()) if not  class_proportions.empty else  0.0



    if min_share  >= min_class_share:
      report["class_balance"] = {
            "passed": True,
            "detail": f"minimum class proportion: {min_share:.2f}"
        }
    else:
      report["class_balance"] = {
            "passed": False,
            "detail": f"minimum class proportion: {min_share:.2f}, below threshold {min_class_share:.2f}"
        }


    # TODO: check 5 - no column has only 1 unique value (zero variance); record passed + detail
    constant_cols= [col for col in df.columns if df[col].nunique(dropna=False) <=1]
    if len(constant_cols) == 0:
        report["zero_variance"] = {
            "passed": True,
            "detail": "No constant columns found"
        }
    else:
        report["zero_variance"] = {
            "passed": False,
            "detail": f"constant column(s) found: {constant_cols}"
        }

    return report


    return report

if __name__ == "__main__":
    df = pd.read_csv(StringIO(BATCH_CSV))
    for check, outcome in report_data_quality(df).items():
        print(check, outcome)



if __name__ == "__main__":
  df = pd.read_csv(StringIO(BATCH_CSV))
  result = report_data_quality(df)

  for check, outcome in result.items():
    status = "PASS" if  outcome['passed'] else  "FAIL"
    print(f"{check}: {status} - {outcome['detail']}")

no_missing_values {'passed': False, 'detail': "1 missing value(s) found in column(s): ['feature_b']"}
feature_count {'passed': True, 'detail': '4 features found, 4 expected'}
binary_target {'passed': True, 'detail': 'target has 2 unique classes'}
class_balance {'passed': True, 'detail': 'minimum class proportion: 0.50'}
zero_variance {'passed': False, 'detail': "constant column(s) found: ['feature_c']"}
no_missing_values: FAIL - 1 missing value(s) found in column(s): ['feature_b']
feature_count: PASS - 4 features found, 4 expected
binary_target: PASS - target has 2 unique classes
class_balance: PASS - minimum class proportion: 0.50
zero_variance: FAIL - constant column(s) found: ['feature_c']
